# 03 - Pipeline smoke test (designed to FINISH)

One question only: **does the pipeline work end to end** - vLLM starts, the
bundle compiles, the agent produces a patch. Nothing about throughput.

Built around two rules learned the hard way:

1. **Finish quickly, whatever happens.** Kaggle discards the log and output of a
   *cancelled* run, so a 2 GPU-hour probe that got cancelled taught us nothing.
   This is 1 task, concurrency 1, about 15 minutes, with a 30-minute hard timeout.
2. **Leave the diagnosis on disk.** Everything printed is appended to
   `/kaggle/working/smoke_log.txt`, and the runner's full stdout and stderr go to
   `smoke_stdout.txt`, so a red cell never costs us the reason.

**Please let it run to the end even if a cell goes red.**

Context is sized from the detected hardware. Anything below the grader's 32768
makes this a pipeline test only, not a comparable measurement.

In [ ]:
import glob
import json
import os
import shutil
import subprocess
import sys
import time
from pathlib import Path

NL = chr(10)
LOG = Path('/kaggle/working/smoke_log.txt')
LOG.parent.mkdir(parents=True, exist_ok=True)


def say(msg=''):
    line = '[' + time.strftime('%H:%M:%S') + '] ' + str(msg)
    print(line, flush=True)
    with LOG.open('a', encoding='utf-8') as fh:
        fh.write(line + NL)


def nvsmi(fields):
    r = subprocess.run(['nvidia-smi', '--query-gpu=' + fields, '--format=csv,noheader'],
                       capture_output=True, text=True)
    return r.stdout.strip()


say('=== GPU ===')
info = nvsmi('name,memory.total,compute_cap')
say(info or '(nvidia-smi returned nothing)')
say('CPUs: ' + str(os.cpu_count()))

rows = [r.split(',') for r in info.splitlines() if r.strip()]
N_GPU = len(rows)
MEM_GB = sum(float(r[1].strip().split()[0]) for r in rows) / 1024 if rows else 0.0
caps = []
for r in rows:
    if len(r) > 2:
        try:
            caps.append(float(r[2].strip()))
        except ValueError:
            pass
CC = min(caps) if caps else 0.0
say('GPUs ' + str(N_GPU) + '  total ' + format(MEM_GB, '.1f') + ' GB  compute cap ' + str(CC))

assert N_GPU >= 2, 'need at least 2 GPUs, got ' + str(N_GPU)

# W4A16 compressed-tensors uses Marlin kernels, which normally need Ampere (8.0+).
# T4 is Turing (7.5). Warn and continue: whether vLLM refuses is what we are testing.
if CC and CC < 8.0:
    say('WARNING compute capability ' + str(CC) + ' is below 8.0; W4A16 Marlin')
    say('        kernels normally need Ampere or newer. vLLM may refuse.')

# 1.21 MB/token measured from the 4xL4 run; weights are ~17 GB, sharded across cards.
KV_GB = MEM_GB * 0.90 - 17.0
FITS = int(KV_GB * 1024 / 1.21) if KV_GB > 0 else 0
MAX_MODEL_LEN = 32768 if FITS >= 32768 else max(4096, (FITS // 2048) * 2048)
say('KV budget ' + format(KV_GB, '.1f') + ' GB -> about ' + format(FITS, ',') + ' tokens')
say('using max_model_len=' + str(MAX_MODEL_LEN))
assert KV_GB > 2, 'only ' + format(KV_GB, '.1f') + ' GB for KV after weights; cannot serve'
if MAX_MODEL_LEN != 32768:
    say('NOTE below the grader 32768 -> PIPELINE TEST, not a comparable measurement')

In [ ]:
for k, v in {
    'LITELLM_LOCAL_MODEL_COST_MAP': 'True',
    'TRANSFORMERS_NO_TF': '1',
    'VLLM_WORKER_MULTIPROC_METHOD': 'spawn',
    'VLLM_ENGINE_READY_TIMEOUT_S': '1200',
    'VLLM_NO_USAGE_STATS': '1',
    'OTEL_SDK_DISABLED': 'true',
    'PYTORCH_CUDA_ALLOC_CONF': 'expandable_segments:True',
}.items():
    os.environ[k] = v

INPUT = Path('/kaggle/input')
say('attached: ' + ', '.join(p.name for p in sorted(INPUT.iterdir())))


def find_dir(needles, must_contain=''):
    hits = []
    for p in INPUT.rglob('*'):
        if not p.is_dir():
            continue
        if any(n in p.name.lower() for n in needles):
            if not must_contain or any(p.glob(must_contain)):
                hits.append(p)
    return sorted(hits, key=lambda q: len(str(q)))[0] if hits else None


def find_model_dir(slug):
    # Kaggle mounts models at .../<instance>/<version>; matching the instance name
    # lands one level above the weights and vLLM rejects it. Anchor on config.json.
    hits = [q.parent for q in INPUT.rglob('config.json') if slug in str(q.parent).lower()]
    return sorted(hits, key=lambda q: len(str(q)))[0] if hits else None


WHEELHOUSE = find_dir(['wheelhouse'], '*.whl')
LAB = find_dir(['swegemma-lab'])
DATA = Path('/kaggle/input/competitions/gemma-4-developer-agent')
MODEL = find_model_dir('gemma-4-31b-it-qat-w4a16-ct')
for name, val in (('wheelhouse', WHEELHOUSE), ('lab', LAB), ('data', DATA), ('model', MODEL)):
    say('  ' + name.ljust(11) + str(val))
assert WHEELHOUSE and LAB and DATA.is_dir() and MODEL, 'missing an attached source'
assert (MODEL / 'config.json').is_file(), 'no config.json in ' + str(MODEL)

In [ ]:
# The wheelhouse is built for ONE Python version. Check before installing:
# the L4 image is 3.12 and works; the T4 image is 3.13, where every compiled
# cp312 wheel is rejected and vLLM's _C.abi3.so then dies with
#   undefined symbol: _ZN3c1013MessageLoggerC1EPKciib
# That cost a 9.5-minute run to discover. This check costs 10 seconds.
import re

running = 'cp' + str(sys.version_info.major) + str(sys.version_info.minor)
tags = set()
for w in WHEELHOUSE.glob('*.whl'):
    tags.update(re.findall(r'-(cp3\d+)-', w.name))
say('running Python tag ' + running + '; wheelhouse tags ' + str(sorted(tags)))
assert not tags or running in tags, (
    'Wheelhouse is built for ' + str(sorted(tags)) + ' but this image runs '
    + running + '. Compiled wheels will be rejected and vLLM will fail to load. '
    'Use the accelerator whose image matches: L4 gives Python 3.12.'
)
say('Python version matches the wheelhouse')

In [ ]:
for pat in ('/usr/local/lib/python*/dist-packages/*cutlass*.pth',
            '/usr/local/lib/python*/site-packages/*cutlass*.pth'):
    for f in glob.glob(pat):
        try:
            os.unlink(f)
        except OSError:
            pass

tmp = Path('/tmp/wheelhouse')
tmp.mkdir(parents=True, exist_ok=True)
for w in WHEELHOUSE.glob('*.whl'):
    if 'cutlass' in w.name.lower():
        continue
    name = w.name
    if 'cu128' in name and '+' not in name:
        name = name.replace('cu128', '+cu128')
    if not (tmp / name).exists():
        os.symlink(w, tmp / name)

wheels = sorted(str(w) for w in tmp.glob('*.whl'))
say('installing ' + str(len(wheels)) + ' wheels')

PIP = [sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', '--force-reinstall']
r = subprocess.run(PIP + wheels, capture_output=True, text=True)
say('bulk pip rc ' + str(r.returncode))

if r.returncode != 0:
    # The official starter installs all wheels in ONE pip call, so a single
    # unsupported wheel aborts the batch and swegemma never lands. Seen on the T4
    # image: apache_tvm_ffi ... is not a supported wheel on this platform.
    say('bulk install failed; retrying per wheel')
    say((r.stderr or '')[-700:])
    failed = []
    for w in wheels:
        rr = subprocess.run(PIP + [w], capture_output=True, text=True)
        if rr.returncode != 0:
            failed.append(Path(w).name)
    say(str(len(failed)) + ' wheel(s) failed individually:')
    for f in failed:
        say('    ' + f)

pre = subprocess.run([sys.executable, str(LAB / 'scripts' / 'calibrate.py'), '--preflight'],
                     capture_output=True, text=True)
say('preflight:')
say((pre.stdout or '') + (pre.stderr or ''))
assert pre.returncode == 0, 'preflight failed - harness imports are broken'

In [ ]:
RUNNER = str(LAB / 'scripts' / 'session_runner.py')
OUT = Path('/kaggle/working/smoke_stdout.txt')

cmd = [sys.executable, RUNNER,
       '--lab', str(LAB), '--data', str(DATA), '--model-path', str(MODEL),
       '--variant', 'v00_public_baseline', '--task-list', 'clean_n.txt',
       '--run-id', 'smoke1', '--limit', '1', '--concurrency', '1',
       '--max-time-minutes', '6', '--max-tool-calls', '20',
       '--max-model-len', str(MAX_MODEL_LEN)]
say('running session_runner with max_model_len=' + str(MAX_MODEL_LEN))

t0 = time.time()
rc = -1
out = ''
err = ''
try:
    proc = subprocess.run(cmd, capture_output=True, text=True, timeout=30 * 60)
    rc = proc.returncode
    out = proc.stdout or ''
    err = proc.stderr or ''
except subprocess.TimeoutExpired as exc:
    rc = -9
    out = exc.stdout if isinstance(exc.stdout, str) else (exc.stdout or b'').decode('utf-8', 'replace')
    err = exc.stderr if isinstance(exc.stderr, str) else (exc.stderr or b'').decode('utf-8', 'replace')
    err = err + NL + '*** TIMEOUT at 30 min ***'
wall = time.time() - t0

parts = ['rc=' + str(rc),
         'wall=' + format(wall, '.0f') + 's',
         'max_model_len=' + str(MAX_MODEL_LEN),
         '', '=== STDOUT ===', out, '', '=== STDERR ===', err]
OUT.write_text(NL.join(parts), encoding='utf-8')
say('rc=' + str(rc) + '  wall=' + format(wall / 60, '.1f') + ' min')
say('--- stdout tail ---')
say(out[-4000:] or '(empty)')
say('--- stderr tail ---')
say(err[-4000:] or '(empty)')

In [ ]:
# Wrapped: a bug in reporting must never hide a successful run.
try:
    base = Path('/kaggle/working/results/smoke1')
    jl = base / 'task_results.jsonl'
    rows = []
    if jl.is_file():
        for line in jl.read_text(encoding='utf-8').splitlines():
            if line.strip():
                rows.append(json.loads(line))
    pdir = base / 'patches'
    patches = sorted(pdir.glob('*.patch')) if pdir.is_dir() else []
    say('=== RESULT ===')
    say('  rows            ' + str(len(rows)))
    say('  patches written ' + str(len(patches)))
    for q in patches:
        say('    ' + q.name + '  ' + str(q.stat().st_size) + ' bytes')
    for row in rows:
        say('  ' + str(row.get('instance_id')) + ': resolved=' + str(row.get('resolved'))
            + ' error=' + str(row.get('error'))[:160])
    ok = bool(patches) and patches[0].stat().st_size > 0
    say('PIPELINE WORKS' if ok else 'NO PATCH PRODUCED - read smoke_stdout.txt')
except Exception as exc:
    say('report cell raised (not fatal): ' + type(exc).__name__ + ': ' + str(exc))

In [ ]:
# Always leave something fetchable. This is the whole point of the redesign.
try:
    for d in ('results', 'ledger'):
        p = Path('/kaggle/working') / d
        if p.is_dir() and any(p.rglob('*')):
            shutil.make_archive('/kaggle/working/smoke_' + d, 'zip', root_dir=p)
            say('archived smoke_' + d + '.zip')
except Exception as exc:
    say('archive raised (not fatal): ' + str(exc))

say('=== files in /kaggle/working ===')
for q in sorted(Path('/kaggle/working').iterdir()):
    size = str(q.stat().st_size) if q.is_file() else '(dir)'
    say('  ' + q.name + '  ' + size)
say('DONE - a cancelled run discards its log, so letting this finish is what matters')